# ICU Patient Deterioration Risk Prediction

This notebook trains a multimodal deep learning model for predicting patient deterioration in the ICU.

**Model Architecture:**
- BiLSTM temporal encoder for vital signs and lab values
- Pre-computed ClinicalBERT embeddings for clinical notes
- Cross-modal attention fusion
- Binary classifier for 24-hour deterioration prediction

**Requirements:**
- GPU runtime (T4 recommended)
- Upload `samples_colab.zip` and `embeddings.zip` to Colab

## 1. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/thedatasense/Patient_Early_Deterioration_Risk_Prediction.git
%cd Patient_Early_Deterioration_Risk_Prediction

# Install dependencies
!pip install -q torch transformers pandas numpy pyarrow scikit-learn pyyaml tqdm matplotlib seaborn

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Upload Data

Upload the following files:
- `samples_colab.zip` (74 MB) - Training data subset
- `embeddings.zip` (142 MB) - ClinicalBERT embeddings

In [ ]:
from google.colab import files

# Upload data files
print("Please upload samples_colab.zip and embeddings.zip")
uploaded = files.upload()

In [ ]:
# Extract uploaded files
import os
os.makedirs("outputs/samples", exist_ok=True)
os.makedirs("outputs/embeddings", exist_ok=True)

!unzip -o samples_colab.zip -d outputs/
!mv outputs/samples_colab/* outputs/samples/
!unzip -o embeddings.zip -d outputs/embeddings/

# Verify extraction
!ls -la outputs/samples/
!ls -la outputs/embeddings/

## 3. Load Data and Model

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from src.data.dataset import ICUDataset, create_dataloaders
from src.models.classifier import create_model
from src.training.losses import create_loss

# Configuration
SAMPLES_DIR = Path("outputs/samples")
EMBEDDINGS_DIR = Path("outputs/embeddings")
OUTPUT_DIR = Path("outputs/models/multimodal")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
BATCH_SIZE = 128  # Larger batch for GPU
EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

In [ ]:
# Create data loaders
print("Creating data loaders...")
train_loader, val_loader, test_loader = create_dataloaders(
    samples_dir=SAMPLES_DIR,
    batch_size=BATCH_SIZE,
    num_workers=2,
    embeddings_dir=EMBEDDINGS_DIR,
)

# Get feature dimensions
train_dataset = train_loader.dataset
n_vitals = train_dataset.n_vitals
n_labs = train_dataset.n_labs

print(f"\nFeatures: {n_vitals} vitals, {n_labs} labs")
print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples: {len(val_loader.dataset):,}")
print(f"Test samples: {len(test_loader.dataset):,}")

In [ ]:
# Model configuration
config = {
    "model": {
        "temporal_encoder": "lstm",
        "lstm": {"hidden_dim": 128, "num_layers": 2, "dropout": 0.3},
        "text": {"projection_dim": 256, "dropout": 0.1},
        "fusion": {"hidden_dim": 128},
        "classifier": {"hidden_dim": 64, "dropout": 0.3},
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 1e-5,
        "max_epochs": EPOCHS,
        "patience": 10,
        "gradient_clip": 1.0,
        "use_amp": True,  # Use AMP on CUDA
        "loss": "focal",
        "focal_loss": {"alpha": 0.25, "gamma": 2.0},
    },
}

# Create model
print("Creating model...")
model = create_model(
    model_type="multimodal",
    n_vitals=n_vitals,
    n_labs=n_labs,
    config=config,
)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## 4. Training

In [ ]:
import time
import json
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, average_precision_score

# Create loss and optimizer
criterion = create_loss("focal", config)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# AMP scaler for mixed precision
scaler = torch.cuda.amp.GradScaler() if DEVICE == "cuda" else None

# Training tracking
best_val_auroc = 0.0
epochs_without_improvement = 0
history = []

print("Starting training...")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        vitals = batch["vitals"].to(device)
        labs = batch["labs"].to(device)
        mask = batch["mask"].to(device)
        static = batch["static"].to(device)
        embedding = batch["embedding"].to(device)
        has_notes = batch["has_notes"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits, _ = model(vitals, labs, mask, static, embedding, has_notes)
                loss = criterion(logits.squeeze(), labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, _ = model(vitals, labs, mask, static, embedding, has_notes)
            loss = criterion(logits.squeeze(), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate model."""
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    for batch in loader:
        vitals = batch["vitals"].to(device)
        labs = batch["labs"].to(device)
        mask = batch["mask"].to(device)
        static = batch["static"].to(device)
        embedding = batch["embedding"].to(device)
        has_notes = batch["has_notes"].to(device)
        labels = batch["label"].to(device)

        logits, _ = model(vitals, labs, mask, static, embedding, has_notes)
        loss = criterion(logits.squeeze(), labels)

        total_loss += loss.item()
        probs = torch.sigmoid(logits.squeeze()).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader)
    auroc = roc_auc_score(all_labels, all_probs)
    auprc = average_precision_score(all_labels, all_probs)

    return avg_loss, auroc, auprc

In [ ]:
# Training loop
PATIENCE = 10

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)

    # Validate
    val_loss, val_auroc, val_auprc = validate(model, val_loader, criterion, DEVICE)

    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    epoch_time = time.time() - epoch_start

    # Record history
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auroc": val_auroc,
        "val_auprc": val_auprc,
        "lr": current_lr,
        "time": epoch_time,
    })

    # Print progress
    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"AUROC: {val_auroc:.4f} | "
          f"AUPRC: {val_auprc:.4f} | "
          f"Time: {epoch_time:.1f}s")

    # Check for improvement
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        epochs_without_improvement = 0
        # Save best model
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_auroc": best_val_auroc,
        }, OUTPUT_DIR / "best_model.pt")
        print(f"  -> New best model saved!")
    else:
        epochs_without_improvement += 1

    # Early stopping
    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nTraining complete! Best Val AUROC: {best_val_auroc:.4f}")

## 5. Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(OUTPUT_DIR / "best_model.pt")
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

# Evaluate on test set
test_loss, test_auroc, test_auprc = validate(model, test_loader, criterion, DEVICE)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"AUROC: {test_auroc:.4f}")
print(f"AUPRC: {test_auprc:.4f}")
print(f"Loss:  {test_loss:.4f}")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = [h["epoch"] for h in history]

# Loss
axes[0].plot(epochs_range, [h["train_loss"] for h in history], label="Train")
axes[0].plot(epochs_range, [h["val_loss"] for h in history], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True)

# AUROC
axes[1].plot(epochs_range, [h["val_auroc"] for h in history])
axes[1].axhline(y=test_auroc, color='r', linestyle='--', label=f"Test: {test_auroc:.4f}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("AUROC")
axes[1].set_title("Validation AUROC")
axes[1].legend()
axes[1].grid(True)

# AUPRC
axes[2].plot(epochs_range, [h["val_auprc"] for h in history])
axes[2].axhline(y=test_auprc, color='r', linestyle='--', label=f"Test: {test_auprc:.4f}")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AUPRC")
axes[2].set_title("Validation AUPRC")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

In [ ]:
# Save results
results = {
    "best_val_auroc": best_val_auroc,
    "test_auroc": test_auroc,
    "test_auprc": test_auprc,
    "test_loss": test_loss,
    "epochs_trained": len(history),
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {OUTPUT_DIR}")

In [ ]:
# Download model and results
from google.colab import files

# Zip outputs
!zip -r model_outputs.zip outputs/models/
files.download("model_outputs.zip")